# Notebook 10: Deutsch-Jozsa Algorithm and Quantum Speedup

This notebook implements the Deutsch-Jozsa algorithm to evaluate black-box oracles in a single quantum query.

---

## Learning Objectives
1. Understand constant versus balanced Boolean functions.
2. Implement phase kickback using an ancilla qubit.
3. Construct the complete Deutsch-Jozsa circuit.
4. Verify deterministic single-query classification.


---
## Real-World Applications & Modern Use Cases

While primarily a foundational theoretical algorithm, the principles of Deutsch-Jozsa underpin:
- **Oracle Complexity & Query Acceleration:** Proving that quantum mechanics provides deterministic exponential speedups over classical algorithms.
- **Hardware Verification of Reversible Logic:** Validating multi-qubit reversible gate arrays in fault-tolerant quantum logic units.


---
## Section 1: Problem Definition

Given a black-box function $f: \{0, 1\}^n \rightarrow \{0, 1\}$ guaranteed to be either constant (same output for all inputs) or balanced (0 for half, 1 for half), determine its type.
- Classical complexity: $2^{n-1} + 1$ queries in the worst case.
- Quantum complexity: Exactly 1 query.


In [1]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler

print("Deutsch-Jozsa environment initialized.")


Deutsch-Jozsa environment initialized.


---
## Section 2: Constructing the Circuit

We initialize $n$ input qubits and 1 ancilla qubit prepared in state $|-\rangle = H|1\rangle$.


In [2]:
def build_deutsch_jozsa(n=3, oracle_type="balanced"):
    qc = QuantumCircuit(n + 1, n)
    
    # Ancilla initialized to |1> then |->
    qc.x(n)
    for q in range(n + 1):
        qc.h(q)
    qc.barrier()
    
    # Oracle implementation
    if oracle_type == "balanced":
        for q in range(n):
            qc.cx(q, n)
    elif oracle_type == "constant":
        pass # f(x) = 0 constant
    qc.barrier()
    
    # Interference layer
    for q in range(n):
        qc.h(q)
        qc.measure(q, q)
        
    return qc

dj_balanced = build_deutsch_jozsa(3, "balanced")
print("Deutsch-Jozsa Circuit (Balanced Oracle):")
print(dj_balanced.draw(output='text'))


Deutsch-Jozsa Circuit (Balanced Oracle):
     ┌───┐      ░                 ░ ┌───┐┌─┐      
q_0: ┤ H ├──────░───■─────────────░─┤ H ├┤M├──────
     ├───┤      ░   │             ░ ├───┤└╥┘┌─┐   
q_1: ┤ H ├──────░───┼────■────────░─┤ H ├─╫─┤M├───
     ├───┤      ░   │    │        ░ ├───┤ ║ └╥┘┌─┐
q_2: ┤ H ├──────░───┼────┼────■───░─┤ H ├─╫──╫─┤M├
     ├───┤┌───┐ ░ ┌─┴─┐┌─┴─┐┌─┴─┐ ░ └───┘ ║  ║ └╥┘
q_3: ┤ X ├┤ H ├─░─┤ X ├┤ X ├┤ X ├─░───────╫──╫──╫─
     └───┘└───┘ ░ └───┘└───┘└───┘ ░       ║  ║  ║ 
c: 3/═════════════════════════════════════╩══╩══╩═
                                          0  1  2


---
## Section 3: Executing Classification

If the input qubits measure all zeros ($|000\rangle$), the function is Constant; otherwise, it is Balanced.


In [3]:
sampler = StatevectorSampler()

for oracle in ["constant", "balanced"]:
    c = build_deutsch_jozsa(3, oracle)
    counts = sampler.run([c], shots=100).result()[0].data.c.get_counts()
    measured_state = list(counts.keys())[0]
    verdict = "Constant" if measured_state == "000" else "Balanced"
    print(f"Oracle Type: {oracle.upper():<10} | Measured: {measured_state} | Classification: {verdict}")


Oracle Type: CONSTANT   | Measured: 000 | Classification: Constant
Oracle Type: BALANCED   | Measured: 111 | Classification: Balanced
